# Laboratório — Algoritmo genético: recarga de veículos elétricos

**Inteligência Artificial aplicada à Engenharia Elétrica · Projeto 3 · 02/12/2026**

Complementa o [simulador](https://furriel.github.io/IA/geneticos/): o algoritmo é implementado do zero (sem biblioteca de otimização), com o mesmo gerador pseudoaleatório, os mesmos operadores e a mesma função objetivo da página. Como a instância é pequena, o ótimo exato é obtido por enumeração e serve de referência.

**Execução:** baixe o `.ipynb` e faça upload no Colab, ou abra no Jupyter. Requer NumPy e Matplotlib.

> Instância didática: eficiência ideal, potência discreta, tarifas fictícias. Não é otimizador operacional.

## Antes de começar

### Objetivo
Executar o algoritmo genético da aula 4, comparar o melhor plano encontrado com o ótimo exato (5,2) e observar o efeito da penalidade λ, da mutação e da semente.

### Contexto do problema
Três veículos elétricos precisam de 4, 3 e 2 kWh em quatro horas, com tarifas diferentes, estação de 4 kW e janelas de conexão. Cada plano é um cromossomo de 12 genes (potência 0, 1 ou 2 kW).

### Pré-requisitos
Nenhum conhecimento prévio de Python. Faça antes, no papel, os cálculos da etapa *Calcular* da aula 4 (custo do plano de 5,2, J com violações, cruzamento).

### Como executar
- **Google Colab:** abra o notebook pelo link da aula, conecte o ambiente e use *Ambiente de execução → Executar tudo*. Para guardar suas alterações, use *Arquivo → Salvar uma cópia no Drive*.
- **Kaggle:** em *kaggle.com/code*, crie um notebook, use *File → Import Notebook* e envie o arquivo `.ipynb`; depois *Run → Run All*.
- **Jupyter local:** requer Python 3 com NumPy e Matplotlib.
- Execute as células **na ordem**; uma célula depende das anteriores.
- Passo a passo ilustrado, com solução de problemas: https://furriel.github.io/IA/apoio/executar-python.html

### Dados e parâmetros
Não há dados externos. Parâmetros no dicionário `CFG`, que você pode alterar **um de cada vez**: `semente`, `populacao`, `geracoes`, `torneio`, `pc`, `pm`, `elite` e `lam`.

## 1. Problema e representação

| Veículo | Energia | Conectado nas horas |
|---|---|---|
| VE1 | 4 kWh | 1–4 |
| VE2 | 3 kWh | 1–3 |
| VE3 | 2 kWh | 2–4 |

Potência por veículo e hora $p_{v,t}\in\{0,1,2\}$ kW; estação $\sum_v p_{v,t}\le 4$ kW; tarifa $c=[1{,}0;\,0{,}7;\,0{,}4;\,0{,}8]$ por kWh. Cromossomo: 12 genes, índice $4v+t$.

$$J = \sum_{v,t} c_t p_{v,t}\Delta t + \lambda_E\,\text{déficit} + \lambda_P\,\text{sobrecarga} + \lambda_J\,\text{energia fora da janela}$$

**Viável** ⇔ déficit = 0, sobrecarga = 0 e nada fora da janela. Uma solução com $J$ pequeno pode ser inviável se as penalidades forem baixas.

In [ ]:
import itertools, json
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

ENERGIA, JANELA = [4, 3, 2], [(0, 3), (0, 2), (1, 3)]
TARIFA, PMAX, DT = [1.0, 0.7, 0.4, 0.8], 4, 1

class LCG:
    def __init__(self, s): self.s = s % 2**32
    def proximo(self):
        self.s = (self.s * 1664525 + 1013904223) % 2**32
        return self.s / 4294967296
    def inteiro(self, n): return int(self.proximo() * n)

def avaliar(crom, lam=10.0):
    custo = deficit = sobre = fora = 0.0
    energia, pot = [], [0, 0, 0, 0]
    for v in range(3):
        e = 0
        for t in range(4):
            p = crom[4 * v + t]; pot[t] += p
            if JANELA[v][0] <= t <= JANELA[v][1]:
                e += p * DT; custo += TARIFA[t] * p * DT
            elif p > 0:
                fora += p * DT; custo += TARIFA[t] * p * DT
        energia.append(e); deficit += max(0, ENERGIA[v] - e)
    for t in range(4): sobre += max(0, pot[t] - PMAX) * DT
    J = custo + lam * deficit + lam * sobre + lam * fora
    return {"J": J, "custo": custo, "deficit": deficit, "sobrecarga": sobre, "fora": fora,
            "viavel": deficit == 0 and sobre == 0 and fora == 0, "energia": energia, "potencia": pot}

base = [2, 2, 0, 0, 2, 1, 0, 0, 0, 1, 1, 0]   # carregar assim que possível, VE1 → VE2 → VE3
print("Linha de base:", avaliar(base))
assert abs(avaliar(base)["custo"] - 7.2) < 1e-9 and avaliar(base)["viavel"]

## 2. Ótimo exato por enumeração ($3^{12}$ = 531 441 soluções)

In [ ]:
todos = np.array(list(itertools.product([0, 1, 2], repeat=12)))
pot = todos.reshape(-1, 3, 4)
mascara = np.array([[JANELA[v][0] <= t <= JANELA[v][1] for t in range(4)] for v in range(3)])
energia = (pot * mascara).sum(2)
viavel = (energia >= np.array(ENERGIA)).all(1) & (pot.sum(1) <= PMAX).all(1) & ((pot * ~mascara).sum((1, 2)) == 0)
custo = (pot * np.array(TARIFA)).sum((1, 2))
otimo = custo[viavel].min()
otimos = todos[viavel][np.isclose(custo[viavel], otimo)]
print(f"Viáveis: {viavel.sum()} de {len(todos)}; custo ótimo = {otimo:.2f}; soluções ótimas empatadas: {len(otimos)}")
print("Exemplo de ótimo:\n", otimos[0].reshape(3, 4), "(linhas = VE1..VE3, colunas = horas, kW)")
assert abs(otimo - 5.2) < 1e-9

## 3. O algoritmo genético (idêntico ao simulador)

Torneio de 3; cruzamento de um ponto ($p_c=0{,}9$); mutação gene a gene ($p_m=0{,}06$) para um nível diferente; elitismo de 2; 60 gerações; população de 30. Ordem de desempate: menor $J$, depois menor índice.

In [ ]:
CFG = dict(semente=20261202, populacao=30, geracoes=60, torneio=3, pc=0.9, pm=0.06, elite=2, lam=10.0)

def executar_ag(cfg=CFG):
    rng = LCG(cfg["semente"])
    pop = [[rng.inteiro(3) for _ in range(12)] for _ in range(cfg["populacao"])]
    hist, pops = [], []
    def resumo(aval, g):
        ordem = sorted(range(len(aval)), key=lambda i: (aval[i]["J"], i))
        viav = [a["custo"] for a in aval if a["viavel"]]
        return {"geracao": g, "melhorJ": aval[ordem[0]]["J"], "melhorCrom": pop[ordem[0]][:], "melhorViavel": aval[ordem[0]]["viavel"],
                "mediaJ": sum(a["J"] for a in aval) / len(aval), "fracaoViavel": len(viav) / len(aval),
                "melhorCustoViavel": min(viav) if viav else None}
    aval = [avaliar(cr, cfg["lam"]) for cr in pop]
    hist.append(resumo(aval, 0)); pops.append([cr[:] for cr in pop])
    for g in range(1, cfg["geracoes"] + 1):
        ordem = sorted(range(len(aval)), key=lambda i: (aval[i]["J"], i))
        novos = [pop[i][:] for i in ordem[:cfg["elite"]]]
        def torneio():
            sorteados = [rng.inteiro(len(pop)) for _ in range(cfg["torneio"])]
            melhor = sorteados[0]
            for s in sorteados:
                if (aval[s]["J"], s) < (aval[melhor]["J"], melhor): melhor = s
            return melhor
        while len(novos) < cfg["populacao"]:
            a, b = pop[torneio()], pop[torneio()]
            f1, f2 = a[:], b[:]
            if rng.proximo() < cfg["pc"]:
                ponto = 1 + rng.inteiro(11)
                f1, f2 = a[:ponto] + b[ponto:], b[:ponto] + a[ponto:]
            for f in (f1, f2):
                for i in range(12):
                    if rng.proximo() < cfg["pm"]:
                        f[i] = (f[i] + 1 + rng.inteiro(2)) % 3
            novos.append(f1)
            if len(novos) < cfg["populacao"]: novos.append(f2)
        pop = novos
        aval = [avaliar(cr, cfg["lam"]) for cr in pop]
        hist.append(resumo(aval, g)); pops.append([cr[:] for cr in pop])
    return hist, pops

hist, pops = executar_ag()
json.dump(hist, open("ag_notebook_resultado.json", "w"))
final = hist[-1]
print(f"Melhor J final = {final['melhorJ']:.2f} (viável: {final['melhorViavel']}); ótimo exato = {otimo:.2f}")
assert all(hist[g]["melhorJ"] <= hist[g - 1]["melhorJ"] + 1e-12 for g in range(1, len(hist)))   # elitismo

## 4. Convergência, população e melhor solução

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
g = [h["geracao"] for h in hist]
ax[0].step(g, [h["melhorJ"] for h in hist], where="post", label="melhor J", color="#146e83")
ax[0].plot(g, [min(h["mediaJ"], 40) for h in hist], ls="--", color="#8aa0ae", label="J médio (cortado em 40)")
ax[0].axhline(otimo, color="#18794e", ls=":", label=f"ótimo exato {otimo:.1f}")
ax[0].axhline(7.2, color="#b42318", ls=":", label="linha de base 7,2")
ax[0].set(title="Evolução do custo", xlabel="geração", ylabel="J (unidades monetárias)", ylim=(0, 40)); ax[0].legend(fontsize=8)
ax[1].plot(g, [100 * h["fracaoViavel"] for h in hist], color="#b45309")
ax[1].set(title="Fração viável da população", xlabel="geração", ylabel="indivíduos viáveis (%)", ylim=(0, 100))
for k, (gg, titulo) in enumerate([(0, "geração 0"), (len(pops) - 1, f"geração {len(pops) - 1}")]):
    pass
im = ax[2].imshow(np.array(pops[-1]), aspect="auto", cmap="Blues", vmin=0, vmax=2)
ax[2].set(title="População final (linhas = indivíduos)", xlabel="gene (VE1 h1…h4 | VE2 | VE3)", ylabel="indivíduo")
fig.colorbar(im, ax=ax[2], label="potência (kW)")
plt.tight_layout(); plt.show()

melhor = final["melhorCrom"]; a = avaliar(melhor)
P = np.array(melhor).reshape(3, 4)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
fundo = np.zeros(4)
for v, cor in enumerate(["#2563a8", "#0f7b74", "#c2570c"]):
    ax[0].bar([f"h{t + 1}\n{TARIFA[t]}/kWh" for t in range(4)], P[v], bottom=fundo, color=cor, label=f"VE{v + 1}")
    fundo += P[v]
ax[0].axhline(PMAX, color="#b42318", ls="--", label="limite 4 kW")
ax[0].set(title="Potência por hora — melhor solução", ylabel="potência (kW)"); ax[0].legend()
ax[1].bar(["VE1", "VE2", "VE3"], ENERGIA, color="#d5dee5", label="necessária")
ax[1].bar(["VE1", "VE2", "VE3"], a["energia"], width=0.5, color="#146e83", label="entregue na janela")
ax[1].set(title=f"Energia por veículo · custo {a['custo']:.2f} · {'VIÁVEL' if a['viavel'] else 'INVIÁVEL'}", ylabel="energia (kWh)"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Cinco execuções independentes

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for s in range(1, 6):
    h, _ = executar_ag({**CFG, "semente": s})
    ating = next((x["geracao"] for x in h if x["melhorCustoViavel"] is not None and abs(x["melhorCustoViavel"] - otimo) < 1e-9), None)
    print(f"semente {s}: melhor custo viável {min(x['melhorCustoViavel'] for x in h if x['melhorCustoViavel'] is not None):.2f}; ótimo na geração {ating}")
    ax.step([x["geracao"] for x in h], [x["melhorJ"] for x in h], where="post", label=f"semente {s}")
ax.axhline(otimo, color="k", ls=":", label="ótimo exato")
ax.set(title="Variabilidade entre sementes", xlabel="geração", ylabel="melhor J", ylim=(4.5, 12)); ax.legend()
plt.tight_layout(); plt.show()

## 6. Solução viável × solução pouco penalizada

Com $\lambda=0{,}3$, deixar de carregar custa menos que carregar: o AG “encontra” $J$ menor que o ótimo, mas a solução não atende os veículos.

In [ ]:
h_baixo, _ = executar_ag({**CFG, "lam": 0.3, "semente": 1})
f = h_baixo[-1]; a = avaliar(f["melhorCrom"], 0.3)
print(f"λ = 0,3: melhor J = {f['melhorJ']:.2f} < ótimo viável {otimo:.2f}, mas viável = {a['viavel']} (déficit {a['deficit']:.0f} kWh)")
assert not a["viavel"] and f["melhorJ"] < otimo

## 7. Exercícios

1. Calcule à mão $J$ da linha de base e de uma solução ótima. Por que a hora 3 é preenchida primeiro?
2. Rode 5 sementes com $p_m = 0$ e com $p_m = 0{,}3$. Explique o efeito sobre diversidade e convergência.
3. Remova o elitismo (`elite = 0`). O melhor $J$ ainda é monótono? Por quê?
4. Encontre o menor $\lambda$ para o qual o melhor indivíduo final é viável nas sementes 1–5. Relacione com a tarifa máxima.
5. Proponha um operador de **reparo** que torne viáveis os filhos e discuta prós e contras em relação à penalização.

Respostas esperadas e rubrica: [Exercícios e rubricas](https://furriel.github.io/IA/apoio/exercicios.html).

## Experimentos propostos
1. Rode as sementes 1 a 5 e compare o melhor custo viável de cada uma com 5,2.
2. Mude `pm` para 0 e para 0,3 e compare a convergência.
3. Reduza `lam` até 0,3 e verifique a viabilidade do melhor indivíduo (seção 6).

## Questões de interpretação
1. O melhor J piora entre gerações? Por quê?
2. Por que, com λ pequeno, o menor J pode ser menor que o ótimo exato e mesmo assim inviável?
3. Uma execução que chega a 5,2 prova que o método sempre encontra o ótimo?

## Conclusão
Registre: o parâmetro alterado (valor antigo → novo), o melhor custo viável e se alcançou 5,2. As respostas esperadas estão em https://furriel.github.io/IA/apoio/gabaritos.html.